In [1]:
from datetime import datetime
from pathlib import Path

from tqdm.auto import tqdm
import xarray as xr

dataset = "era5_land"

save_storage = Path("../data/MeteoData/ParsedMonthly") / dataset
save_storage.mkdir(parents=True, exist_ok=True)

years = range(2007, 2025)

for year in tqdm(years, desc="Processing years"):
    with xr.open_mfdataset(
        list(Path("../data/MeteoData/parsed_meteo/era5_land/").rglob(f"*/{year}.nc")),
        combine="by_coords",
        chunks={"time": 30},
    ) as ds:
        ds = ds.sortby("time")
        for month in tqdm(range(1, 13), desc=f"Processing months of {year}"):
            ds_month = ds.sel(time=ds.time.dt.month == month)
            save_path = save_storage / f"{year:04d}_{month:02d}.nc"
            if save_path.exists():
                continue
            ds_month.to_netcdf(save_path)


Processing years:   0%|          | 0/18 [00:00<?, ?it/s]

Processing months of 2007:   0%|          | 0/12 [00:00<?, ?it/s]

Processing months of 2008:   0%|          | 0/12 [00:00<?, ?it/s]

Processing months of 2009:   0%|          | 0/12 [00:00<?, ?it/s]

Processing months of 2010:   0%|          | 0/12 [00:00<?, ?it/s]

...

Processing months of 2020:   0%|          | 0/12 [00:00<?, ?it/s]

Processing months of 2021:   0%|          | 0/12 [00:00<?, ?it/s]

Processing months of 2022:   0%|          | 0/12 [00:00<?, ?it/s]

Processing months of 2023:   0%|          | 0/12 [00:00<?, ?it/s]

Processing months of 2024:   0%|          | 0/12 [00:00<?, ?it/s]

In [ ]:
dataset = "mswep"

save_storage = Path("/media/dmbrmv/ssd_2b/MSWEP/ParsedMonthly") / dataset
save_storage.mkdir(parents=True, exist_ok=True)
years = range(1979, 2025)
data_dir = Path("/media/dmbrmv/ssd_2b/MSWEP/data")


def files_for_month(year, month, data_dir=data_dir):
    # compute day-of-year range for the given month
    start = datetime(year, month, 1)
    if month < 12:
        start_next = datetime(year, month + 1, 1)
    else:
        start_next = datetime(year + 1, 1, 1)
    start_doy = start.timetuple().tm_yday
    end_doy = (start_next - datetime(year, 1, 1)).days

    matches = []
    for p in data_dir.rglob("*.nc"):
        stem = p.stem  # expected format: 'YYYYDDD' e.g. '1980364'
        if not stem.startswith(str(year)):
            continue
        doy_part = stem[len(str(year)) :]
        if not doy_part.isdigit():
            continue
        doy = int(doy_part)
        if start_doy <= doy <= end_doy:
            matches.append((doy, p))

    matches.sort(key=lambda x: x[0])  # sort by day-of-year
    return [p for doy, p in matches]


month = 2
year = 2020

for year in tqdm(years, desc="Processing years"):
    for month in tqdm(range(1, 13), desc=f"Processing months of {year}"):
        save_path = save_storage / f"{year:04d}_{month:02d}.nc"
        if save_path.exists():
            continue
        files = files_for_month(year, month)
        if not files:
            print(f"No files found for {year}-{month:02d}, skipping.")
            continue
        with xr.open_mfdataset(files, combine="by_coords", chunks={"time": 30}) as ds:
            ds = ds.sortby("time")
            ds = (
                ds.where(ds.lat > 41.0, drop=True)
                .where(ds.lat < 82.0, drop=True)
                .where(ds.lon > 19.0, drop=True)
                .where(ds.lon < 179.0, drop=True)
            )
            ds.to_netcdf(save_path)